# Reproduce: Who the map misses

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mugeha-laura/bias-bounty-overture-gaps/blob/main/reproduce.ipynb)

This notebook reproduces the main results of our entry to the Zindi **Bias Bounty Mapping Equity Challenge** in about a minute, with no setup:

1. Rebuild the two final submissions (`v34`, `v15`) and confirm they match the uploaded files exactly.
2. Turn the public leaderboard scores into each signal's correlation with the truth.
3. Show the Best Bias Discovery: fire departments a category search cannot find.

It uses the features committed to the repository. To rebuild them from the raw challenge data (~20 minutes), run the last cell.
Method and sources: [docs/](https://github.com/mugeha-laura/bias-bounty-overture-gaps/tree/main/docs).

## Setup
Gets the repository (when not already inside it) and installs DuckDB. pandas and NumPy come preinstalled on Colab; results are identical across pandas 2.2–3.0.

In [1]:
import os, subprocess, sys
REPO = "https://github.com/mugeha-laura/bias-bounty-overture-gaps.git"
if not os.path.exists("score.py"):
    if not os.path.exists("bias-bounty-overture-gaps"):
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    os.chdir("bias-bounty-overture-gaps")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb>=1.3"], check=True, capture_output=True)

def run(*args):
    """Run a repository script and show its output."""
    out = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(out.stdout + out.stderr)
    if out.returncode != 0:
        raise RuntimeError(f"{' '.join(args)} failed")

print("ready in", os.path.basename(os.getcwd()))

ready in nbrun


## 1. Rebuild the final submissions
Each version in `score.py` rebuilds by name. We keep the uploaded files aside, rebuild, and compare. If `SampleSubmission.csv` from Zindi is not here, the scripts use the same tract list from the committed submission.

In [2]:
import shutil, tempfile
tmp = tempfile.gettempdir()   # works on Colab, macOS, Linux and Windows
for v in ["v34", "v15"]:
    shutil.copy(f"submissions/submission_{v}.csv", os.path.join(tmp, f"uploaded_{v}.csv"))
    run("score.py", v)
    run("check_submission.py", f"submissions/submission_{v}.csv", "--same-as", os.path.join(tmp, f"uploaded_{v}.csv"))

note: SampleSubmission.csv not found; using the tract list from submissions/submission_v34.csv
submission_v34.csv: 9794 rows, mean score 0.060



note: SampleSubmission.csv not found; using the tract list from submissions/submission_v34.csv
OK: 9794 tracts, mean score 0.0600



note: SampleSubmission.csv not found; using the tract list from submissions/submission_v34.csv
submission_v15.csv: 9794 rows, mean score 0.060



note: SampleSubmission.csv not found; using the tract list from submissions/submission_v34.csv
OK: 9794 tracts, mean score 0.0598



## 2. What the leaderboard told us
Two constant submissions fix the target's mean (0.0598) and spread (0.0890). Each probe then moves every tract by one signal, and its score gives that signal's correlation with the truth (`corr_with_truth`). Lower `public_rmse` is better among models (v10, v15, v20, v34); for probes, the correlation is the result.

In [3]:
run("analyse.py")

public target: mean 0.0598, sd 0.0890

version  public_rmse  pred_mean    bias  corr_with_truth
     v1       0.1314     0.1153  0.0555          -0.1288
     v2       0.1422     0.0988  0.0390          -0.1865
     v3       0.1313     0.1168  0.0570          -0.1076
     v4       0.1445     0.1215  0.0617          -0.1334
     v5       0.0890     0.0598 -0.0000              NaN
     v6       0.0835     0.0598 -0.0000           0.3484
     v7       0.0909     0.0598 -0.0000           0.1039
     v8       0.1023     0.0598 -0.0000          -0.3080
     v9       0.0926     0.0610  0.0012          -0.0082
    v10       0.0821     0.0598 -0.0000           0.3879
    v11       0.0818     0.0598  0.0000           0.4265
    v12       0.0859     0.0611  0.0013           0.2915
    v13       0.0887     0.0598  0.0000           0.3382
    v14       0.0927     0.0599  0.0001           0.1075
    v15       0.0787     0.0598  0.0000           0.4664
    v16       0.0847     0.0601  0.0003          

## 3. Best Bias Discovery: fire departments a category search cannot find
A place counts as hidden when its name says fire department, its category is not `fire_department`, and no `fire_department` place lies within 500 m. Intervals resample whole counties.

In [4]:
import pandas as pd
summary = pd.read_csv("discovery/hidden_fire_summary.csv")
summary[["departments", "hidden"]] = summary[["departments", "hidden"]].astype("Int64")   # whole numbers
pct = lambda x: f"{x * 100:.2f}%"
summary["share hidden"] = summary.share.map(pct)
summary["95% CI"] = summary.ci_low.map(pct) + " – " + summary.ci_high.map(pct)
summary[["group", "departments", "hidden", "share hidden", "95% CI", "median_km_to_findable"]]

,group,departments,hidden,share hidden,95% CI,median_km_to_findable
0,Other rural,1470,367,24.97%,22.64% – 27.45%,11.1579
1,Rural tribal Eastern OK,466,123,26.39%,21.60% – 31.42%,10.0085
2,Urban,2402,207,8.62%,7.45% – 10.19%,2.3700
3,Rural minus urban,<NA>,<NA>,16.69%,13.95% – 19.22%,NaN


Named examples in Oklahoma, with the category Overture files them under and the distance to the nearest station a category search would find:

In [5]:
hidden = pd.read_csv("discovery/hidden_fire_departments.csv", dtype={"GEOID": str})
examples = ["Rock Volunteer Fire Department", "Prague Volunteer Fire Department", "Putnam Volunteer Fire Department",
            "Fairfax Volunteer Fire Department", "Eldorado Fire Department"]
hidden[(hidden.region == "eastern-ok") & hidden.name.isin(examples)][["name", "category", "GEOID", "nearest_fire_dept_km"]]

,name,category,GEOID,nearest_fire_dept_km
0,Eldorado Fire Department,fire_protection_service,40065968300,36.8822
5,Fairfax Volunteer Fire Department,community_services_non_profits,40113940003,23.9066
7,Putnam Volunteer Fire Department,professional_services,40043959300,23.7850
64,Prague Volunteer Fire Department,central_government_office,40081961500,10.8379
122,Rock Volunteer Fire Department,physical_therapy,40113940007,5.8462


Validation against OpenStreetMap (the one outside dataset, used in the discovery only), plus the other checks:

In [6]:
pd.read_csv("discovery/hidden_fire_extra.csv").round(3)

,check,group,n,value,note
0,osm,"hidden, rural",490,0.280,share with an OSM amenity=fire_station within ...
1,osm,"hidden, urban",207,0.435,share with an OSM amenity=fire_station within ...
2,osm,"hidden, rural tribal Eastern OK",123,0.089,share with an OSM amenity=fire_station within ...
3,osm,"hidden, all",697,0.326,share with an OSM amenity=fire_station within ...
4,osm,"control: fire_department, all",3641,0.549,share with an OSM amenity=fire_station within ...
5,hazard,"rural, wildfire hazard Q1 lowest",484,0.238,share of fire departments hidden
6,hazard,"rural, wildfire hazard Q2",485,0.245,share of fire departments hidden
7,hazard,"rural, wildfire hazard Q3",483,0.248,share of fire departments hidden
8,hazard,"rural, wildfire hazard Q4 highest",484,0.281,share of fire departments hidden
9,hazard,"rural, summer drought Q1 lowest",486,0.239,share of fire departments hidden


## Optional: rebuild everything from the raw challenge data
Reads the challenge bucket directly; about 20 minutes for features and 5 for the discovery. Uncomment to run.

In [7]:
# run("features.py", "all")
# run("strata_extra.py")
# run("discovery_fire.py")
# run("discovery_extra.py")